# 🏥⚖️ QLoRA Fine-tuning — Médico & Abogado
Fine-tuning de Qwen2.5-7B-Instruct sobre dataset médico-legal.
Setup: 2x T4, DDP, QLoRA 4-bit durante entrenamiento.

In [1]:
# ─────────────────────────────────────────────
# CELDA 1 — Verificar entorno y GPUs
# ─────────────────────────────────────────────
import os
import gc
import torch

print(f"🖥️  GPUs disponibles: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"   GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB total | {free/1e9:.1f} GB libres")

assert torch.cuda.device_count() >= 2, "❌ Necesitás 2 GPUs. Activá T4x2 en Settings."
print("\n✅ Entorno listo para DDP")


🖥️  GPUs disponibles: 2
   GPU 0: Tesla T4 — 15.6 GB total | 15.5 GB libres
   GPU 1: Tesla T4 — 15.6 GB total | 15.5 GB libres

✅ Entorno listo para DDP


In [2]:
import sys, os

os.system(f"{sys.executable} -m pip install -q --force-reinstall --no-deps "
          "transformers==4.46.3 "
          "tokenizers==0.20.3 "
          "huggingface-hub==0.26.5 "
          "trl==0.11.4 "
          "accelerate==0.27.2 "
          "peft==0.13.2 ")

os.system(f"{sys.executable} -m pip install -q lm-eval bitsandbytes datasets")

import importlib
for pkg in ["transformers", "tokenizers", "huggingface_hub", "trl", "accelerate", "peft"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"✅ {pkg}=={mod.__version__}")
    except Exception as e:
        print(f"❌ {pkg}: {e}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.8/447.8 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 24.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

2026-03-07 19:09:32.829321: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772910573.021563      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772910573.077075      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772910573.502264      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772910573.502305      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772910573.502308      55 computation_placer.cc:177] computation placer alr

✅ peft==0.13.2


In [3]:
import os

accelerate_config = """compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
mixed_precision: 'fp16'
num_machines: 1
num_processes: 2
"""

os.makedirs("/root/.cache/huggingface/accelerate", exist_ok=True)
with open("/root/.cache/huggingface/accelerate/default_config.yaml", "w") as f:
    f.write(accelerate_config)

print("✅ Accelerate config: DDP, mixed precision fp16")


✅ Accelerate config: DDP, mixed precision fp16


In [4]:
# ─────────────────────────────────────────────
# CELDA 4 — Cargar dataset médico + legal
# ─────────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets
import random

print("📦 Cargando dataset médico (HealthCareMagic 100k)...")
medical = load_dataset(
    "lavita/ChatDoctor-HealthCareMagic-100k",
    split="train",
    streaming=False
)

print("📦 Cargando dataset legal (Law Stack Exchange)...")
legal = load_dataset(
    "jonathanli/law-stack-exchange",
    split="train",
    streaming=False
)

# Balancear: 30k médico + 15k legal = 45k total
N_MEDICAL = 18_000
N_LEGAL   = 9_000

medical = medical.shuffle(seed=42).select(range(min(N_MEDICAL, len(medical))))
legal   = legal.shuffle(seed=42).select(range(min(N_LEGAL, len(legal))))

print(f"✅ Médico: {len(medical):,} ejemplos")
print(f"✅ Legal:  {len(legal):,} ejemplos")
print(f"\n🔍 Columnas médico: {medical.column_names}")
print(f"🔍 Columnas legal:  {legal.column_names}")
print(f"\n🔍 Ejemplo médico:")
print(medical[0])
print(f"\n🔍 Ejemplo legal:")
print(legal[0])


📦 Cargando dataset médico (HealthCareMagic 100k)...


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

(…)-00000-of-00001-5e7cb295b9cff0bf.parquet:   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

📦 Cargando dataset legal (Law Stack Exchange)...


...

test.jsonl:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/638 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/319 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1596 [00:00<?, ? examples/s]

✅ Médico: 18,000 ejemplos
✅ Legal:  638 ejemplos

🔍 Columnas médico: ['instruction', 'input', 'output']
🔍 Columnas legal:  ['Id', 'PostTypeId', 'CreationDate', 'Score', 'ViewCount', 'LastActivityDate', 'AnswerCount', 'CommentCount', 'ContentLicense', 'body', 'text_label', 'title']

🔍 Ejemplo médico:
{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'I have been having alot of catching ,pain and discomfort under my right rib.  If I twist to either side especially my right it feels like my rib actually catches on something and at times I have to stop try to catch my breath and wait for it to subside.  There are times if I am laughing too hard that it will do the same thing but normally its more so if I have twisted or moved  a certain way', 'output': 'Hi thanks for asking question. Here you are complaining pain in particular position esp. While turning to a side. So strong possibility is about moderate degree muscular

In [5]:
# ─────────────────────────────────────────────
# CELDA 5 — Formatear en ChatML con system prompt
# ─────────────────────────────────────────────
from datasets import concatenate_datasets

SYSTEM_MEDICO = (
    "Sos un médico clínico experto con más de 20 años de experiencia. "
    "Respondés preguntas médicas de forma clara, precisa y empática. "
    "Siempre recomendás consultar a un profesional de la salud para diagnósticos definitivos."
)

SYSTEM_LEGAL = (
    "Sos un abogado experto en derecho con más de 20 años de experiencia. "
    "Respondés consultas legales de forma clara y precisa. "
    "Siempre aclarás que tu respuesta es orientativa y recomendás consultar a un abogado matriculado."
)

def format_medical(example):
    instruction = example.get("instruction", "").strip()
    inp         = example.get("input", "").strip()
    output      = example.get("output", "").strip()
    if not instruction or not output:
        return {"text": ""}
    user_msg = f"{instruction}\n{inp}".strip() if inp else instruction
    text = (
        f"<|im_start|>system\n{SYSTEM_MEDICO}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n{output}<|im_end|>"
    )
    return {"text": text}

def format_legal(example):
    # jonathanli/law-stack-exchange tiene: title, question, answer
    question = (example.get("title", "") + "\n" + example.get("question", "")).strip()
    answer   = example.get("answer", "").strip()
    if not question or not answer:
        return {"text": ""}
    text = (
        f"<|im_start|>system\n{SYSTEM_LEGAL}<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )
    return {"text": text}

medical_fmt = medical.map(format_medical, remove_columns=medical.column_names, num_proc=4)
legal_fmt   = legal.map(format_legal,   remove_columns=legal.column_names,   num_proc=4)

# Filtrar vacíos y muy cortos
medical_fmt = medical_fmt.filter(lambda x: len(x["text"]) > 100)
legal_fmt   = legal_fmt.filter(lambda x: len(x["text"]) > 100)

# Combinar y mezclar
dataset = concatenate_datasets([medical_fmt, legal_fmt]).shuffle(seed=42)
dataset = dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

print(f"✅ Dataset combinado formateado")
print(f"📊 Train: {len(train_dataset):,} | Val: {len(eval_dataset):,}")
print(f"\n🔍 Ejemplo médico formateado:")
print(train_dataset[0]["text"][:600])
print("...")


Map (num_proc=4):   0%|          | 0/18000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/638 [00:00<?, ? examples/s]

Filter:   0%|          | 0/18000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/638 [00:00<?, ? examples/s]

✅ Dataset combinado formateado
📊 Train: 17,100 | Val: 900

🔍 Ejemplo médico formateado:
<|im_start|>system
Sos un médico clínico experto con más de 20 años de experiencia. Respondés preguntas médicas de forma clara, precisa y empática. Siempre recomendás consultar a un profesional de la salud para diagnósticos definitivos.<|im_end|>
<|im_start|>user
If you are a doctor, please answer the medical questions based on the patient's description.
Hello Doctor        I\"ve just been diagnosed with malignant tumor on lower esophagus(small in size) Have not had petyet.But trying to prepare aplan. Realisticaly how painful and uncomfortable.Is the dying process and does the removal of esoph
...


In [6]:
import torch
from transformers import AutoTokenizer

MODEL_ID   = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "/kaggle/working/qwen25-7b-medlegal"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True, padding_side="right"
)
tokenizer.pad_token = "<|endoftext|>"

print(f"✅ Tokenizer listo")
print(f"   pad_token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"   eos_token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer listo
   pad_token: <|endoftext|> (id=151643)
   eos_token: <|im_end|> (id=151645)


In [7]:
# ─────────────────────────────────────────────
# CELDA 7 — Guardar dataset en disco para DDP
# ─────────────────────────────────────────────
DATA_PATH = "/kaggle/working/medlegal_formatted"
dataset.save_to_disk(DATA_PATH)
print(f"✅ Dataset guardado en {DATA_PATH}")


Saving the dataset (0/1 shards):   0%|          | 0/17100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/900 [00:00<?, ? examples/s]

✅ Dataset guardado en /kaggle/working/medlegal_formatted


In [8]:
# ─────────────────────────────────────────────
# CELDA 8 — Pre-descargar modelo (evita race condition DDP)
# ─────────────────────────────────────────────
from huggingface_hub import snapshot_download
import torch, gc

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print("📥 Descargando archivos del modelo a caché local...")
print("   (sin cargar en memoria ni tocar VRAM)")

snapshot_download(
    repo_id=MODEL_ID,
    repo_type="model",
    ignore_patterns=["*.msgpack", "*.h5", "flax_*"],
)

print("✅ Modelo en caché — los workers DDP leerán desde disco")
print(f"   VRAM libre: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / {torch.cuda.mem_get_info()[1]/1e9:.1f} GB")


📥 Descargando archivos del modelo a caché local...
   (sin cargar en memoria ni tocar VRAM)


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

✅ Modelo en caché — los workers DDP leerán desde disco
   VRAM libre: 15.5 GB / 15.6 GB


In [9]:
train_script = '''
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_from_disk

MODEL_ID   = "Qwen/Qwen2.5-7B-Instruct"
DATA_PATH  = "/kaggle/working/medlegal_formatted"
OUTPUT_DIR = "/kaggle/working/qwen25-7b-medlegal"

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

def main():
    local_rank = int(os.environ.get("LOCAL_RANK", 0))

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID, trust_remote_code=True, padding_side="right"
    )
    tokenizer.pad_token = "<|endoftext|>"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        trust_remote_code=True,
        device_map={"": local_rank},
        attn_implementation="sdpa",
    )

    model = prepare_model_for_kbit_training(model)
    model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=32,                              # ✅ era 64, bajamos para caber en T4
        lora_alpha=64,                     # ✅ siempre el doble de r
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.config.use_cache = False

    if local_rank == 0:
        model.print_trainable_parameters()

    dataset    = load_from_disk(DATA_PATH)
    train_data = dataset["train"]
    eval_data  = dataset["test"]

    sft_config = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=2,
        per_device_train_batch_size=1,     # ✅ era 2, bajamos para ganar ~2GB
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,    # ✅ era 8, batch efectivo sigue siendo 32
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.10,
        weight_decay=0.01,
        fp16=True,
        bf16=False,
        max_grad_norm=0.3,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        max_seq_length=768,                # ✅ era 1024, punto medio VRAM/calidad
        packing=True,
        dataset_text_field="text",
        report_to="none",
        dataloader_num_workers=4,
        ddp_find_unused_parameters=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_data,
        eval_dataset=eval_data,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    if local_rank == 0:
        sample_batch = next(iter(trainer.get_train_dataloader()))
        labels = sample_batch["labels"]
        valid  = (labels != -100).sum().item()
        total  = labels.numel()
        print(f"\\\\n🔍 Sanity check — labels válidos: {valid}/{total} ({valid/total*100:.1f}%)")
        if valid == 0:
            raise ValueError("❌ TODOS los labels son -100!")

        batch_efectivo = sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps * 2
        steps_totales  = len(train_data) // batch_efectivo
        print(f"\\\\n🚀 Iniciando fine-tuning QLoRA en 2xT4...")
        print(f"   Batch efectivo: {batch_efectivo} | Steps estimados: {steps_totales}")
        print(f"   ⚡ Packing=True — steps reales serán menos\\\\n")

    trainer.train()

    if local_rank == 0:
        print("\\\\n💾 Guardando adapters LoRA...")
        trainer.model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"\\\\n✅ Adapters guardados en {OUTPUT_DIR}")

if __name__ == "__main__":
    main()
'''

with open("/kaggle/working/train_ddp.py", "w") as f:
    f.write(train_script)

print("✅ train_ddp.py guardado")

✅ train_ddp.py guardado


In [10]:
import os, shutil, gc, torch

os.environ["TOKENIZERS_PARALLELISM"] = "false"

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"   GPU {i}: {free/1e9:.1f} GB libres / {total/1e9:.1f} GB totales")

ckpt_dir = "/kaggle/working/qwen25-7b-medlegal"
if os.path.exists(ckpt_dir):
    shutil.rmtree(ckpt_dir)
    print(f"🧹 Output dir limpiado: {ckpt_dir}")

print("🚀 Lanzando entrenamiento con accelerate + DDP...\n")
exit_code = os.system("accelerate launch /kaggle/working/train_ddp.py")

if exit_code == 0:
    print("\n✅ Entrenamiento completado exitosamente")
else:
    print(f"\n❌ Error (exit code: {exit_code}) — revisá los logs arriba")


   GPU 0: 15.5 GB libres / 15.6 GB totales
   GPU 1: 15.5 GB libres / 15.6 GB totales
🚀 Lanzando entrenamiento con accelerate + DDP...



2026-03-07 19:12:37.562094: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-07 19:12:37.562306: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772910757.583405     236 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772910757.583754     237 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772910757.590087     236 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1772910757.590518     237 cuda_blas.cc:1

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


Generating train split: 7026 examples [00:15, 464.85 examples/s]
Generating train split: 369 examples [00:00, 448.72 examples/s]
/usr/local/lib/python3.12/dist-packages/torch/distributed/distributed_c10d.py:4876: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  warnings.warn(  # warn only once
[rank0]:[W307 19:14:08.590855688 ProcessGroupNCCL.cpp:5068] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler(

\n🔍 Sanity check — labels válidos: 768/768 (100.0%)
\n🚀 Iniciando fine-tuning QLoRA en 2xT4...
   Batch efectivo: 32 | Steps estimados: 534
   ⚡ Packing=True — steps reales serán menos\n


...

{'loss': 1.6656, 'grad_norm': 0.36833900213241577, 'learning_rate': 3.8440076345561684e-07, 'epoch': 1.96}


100%|██████████| 438/438 [7:27:28<00:00, 60.08s/it]/usr/local/lib/python3.12/dist-packages/torch/distributed/distributed_c10d.py:4876: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  warnings.warn(  # warn only once
100%|██████████| 438/438 [7:27:29<00:00, 61.30s/it]


{'train_runtime': 26850.2516, 'train_samples_per_second': 0.523, 'train_steps_per_second': 0.016, 'train_loss': 1.7264029859952186, 'epoch': 2.0}
\n💾 Guardando adapters LoRA...
\n✅ Adapters guardados en /kaggle/working/qwen25-7b-medlegal


[rank0]:[W308 02:41:42.202706363 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



✅ Entrenamiento completado exitosamente


In [11]:
# ─────────────────────────────────────────────
# CELDA 11 — Merge LoRA al modelo base
# ─────────────────────────────────────────────
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID    = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_DIR = "/kaggle/working/qwen25-7b-medlegal"
OUTPUT_DIR  = "/kaggle/working/qwen25-7b-medlegal-merged"

gc.collect()
torch.cuda.empty_cache()
print(f"🧹 VRAM libre antes del merge: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

print("📥 Cargando modelo base en fp16...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("🔀 Cargando y mergeando adapters LoRA...")
model_merged = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model_merged = model_merged.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)

print(f"💾 Guardando modelo mergeado en {OUTPUT_DIR}...")
model_merged.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Merge completado → {OUTPUT_DIR}")


🧹 VRAM libre antes del merge: 15.4 GB
📥 Cargando modelo base en fp16...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔀 Cargando y mergeando adapters LoRA...
💾 Guardando modelo mergeado en /kaggle/working/qwen25-7b-medlegal-merged...
✅ Merge completado → /kaggle/working/qwen25-7b-medlegal-merged


In [12]:
# ─────────────────────────────────────────────
# CELDA 12 — Guardar modelo en Hugging Face Hub
# ─────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

REPO_ID    = "Cukinator/qwen25-7b-medlegal"
OUTPUT_DIR = "/kaggle/working/qwen25-7b-medlegal-merged"

api = HfApi()
api.create_repo(repo_id=REPO_ID, private=True, exist_ok=True)

print(f"📤 Subiendo archivos desde {OUTPUT_DIR}...")
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    repo_type="model",
)
print(f"✅ Guardado en https://huggingface.co/{REPO_ID}")


📤 Subiendo archivos desde /kaggle/working/qwen25-7b-medlegal-merged...


model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

Upload 5 LFS files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

✅ Guardado en https://huggingface.co/Cukinator/qwen25-7b-medlegal


In [13]:
# ─────────────────────────────────────────────
# CELDA 13 — Verificar archivos guardados
# ─────────────────────────────────────────────
import os

OUTPUT_DIR = "/kaggle/working/qwen25-7b-medlegal-merged"
files = os.listdir(OUTPUT_DIR) if os.path.exists(OUTPUT_DIR) else []

print(f"📁 Archivos en {OUTPUT_DIR}:")
total_gb = 0
for f in sorted(files):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    total_gb += size_mb / 1000
    print(f"   {f:45s} {size_mb:>8.1f} MB")
print(f"\n   Total: {total_gb:.2f} GB")


📁 Archivos en /kaggle/working/qwen25-7b-medlegal-merged:
   added_tokens.json                                  0.0 MB
   config.json                                        0.0 MB
   generation_config.json                             0.0 MB
   merges.txt                                         1.7 MB
   model-00001-of-00004.safetensors                4877.7 MB
   model-00002-of-00004.safetensors                4932.8 MB
   model-00003-of-00004.safetensors                4330.9 MB
   model-00004-of-00004.safetensors                1090.0 MB
   model.safetensors.index.json                       0.0 MB
   special_tokens_map.json                            0.0 MB
   tokenizer.json                                    11.4 MB
   tokenizer_config.json                              0.0 MB
   vocab.json                                         2.8 MB

   Total: 15.25 GB


In [14]:
# ─────────────────────────────────────────────
# CELDA 14 — Cargar modelo desde Hugging Face Hub
# ─────────────────────────────────────────────
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

REPO_ID = "Cukinator/qwen25-7b-medlegal"

print(f"📥 Cargando modelo desde {REPO_ID}...")
tokenizer_eval = AutoTokenizer.from_pretrained(REPO_ID, trust_remote_code=True)
model_eval = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model_eval.eval()
print("✅ Modelo cargado correctamente")
print(f"   VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB")


📥 Cargando modelo desde Cukinator/qwen25-7b-medlegal...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Modelo cargado correctamente
   VRAM usada: 12.9 GB


In [15]:
# ─────────────────────────────────────────────
# CELDA 15 — Probar el modelo médico-legal
# ─────────────────────────────────────────────
from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation",
    model=model_eval,
    tokenizer=tokenizer_eval,
    torch_dtype=torch.float16,
    device_map="auto"
)

SYSTEM_MEDICO = (
    "Sos un médico clínico experto con más de 20 años de experiencia. "
    "Respondés preguntas médicas de forma clara, precisa y empática. "
    "Siempre recomendás consultar a un profesional de la salud para diagnósticos definitivos."
)

SYSTEM_LEGAL = (
    "Sos un abogado experto en derecho con más de 20 años de experiencia. "
    "Respondés consultas legales de forma clara y precisa. "
    "Siempre aclarás que tu respuesta es orientativa y recomendás consultar a un abogado matriculado."
)

test_cases = [
    (SYSTEM_MEDICO, "Tengo fiebre de 38.5°C hace 3 días con dolor de garganta, ¿qué puede ser?"),
    (SYSTEM_MEDICO, "¿Cuál es la diferencia entre diabetes tipo 1 y tipo 2?"),
    (SYSTEM_LEGAL,  "Mi empleador no me pagó el sueldo de este mes, ¿qué puedo hacer?"),
    (SYSTEM_LEGAL,  "¿Qué diferencia hay entre un contrato laboral y uno de locación de servicios?"),
]

print("🗣️ Respuestas del modelo médico-legal:\n")
for system, prompt in test_cases:
    role = "🏥 MÉDICO" if "médico" in system else "⚖️ LEGAL"
    messages = [
        {"role": "system",    "content": system},
        {"role": "user",      "content": prompt},
    ]
    output = pipe(
        messages,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer_eval.eos_token_id
    )
    response = output[0]["generated_text"][-1]["content"]
    print(f"{role} ❓ {prompt}")
    print(f"💬 {response}")
    print("-" * 60)


🗣️ Respuestas del modelo médico-legal:

🏥 MÉDICO ❓ Tengo fiebre de 38.5°C hace 3 días con dolor de garganta, ¿qué puede ser?
💬 Hello dear, thanks for your question on Chat Doctor. I can understand your concern. By your history and description, possibility of viral sore throat is more. But better to rule out streptococcal infection (strep throat) first. So get done throat swab culture and sensitivity report. If it is negative then you are having viral sore throat. And for this you don't need any antibiotics. Just take painkiller and anti-inflammatory Chat Doctor.  Do warm saline gargles.  Avoid hot and spicy food.  Take soft and bland diet. Don't worry, you will be alright with all these. Hope I have solved your query. Wish you good health. Thanks.
------------------------------------------------------------
🏥 MÉDICO ❓ ¿Cuál es la diferencia entre diabetes tipo 1 y tipo 2?
💬 Hola. Gracias por consultar en Chat Doctor. Diabeticos tipo 1 son diabetes autoinmunes. Es decir, se produce una 